# 中证800 V46 Label 单变量对照实验

目的：保持 V46 的特征、参数、训练窗口、JoinQuant pkl 协议不变，只改变训练 target，比较 label 本身对线上回测的影响。

本轮只保留已经有价值的原始/中位数 label，再新增一个更贴近线上成交口径的单变量：

1. `alpha_1m`：原始 V46 label，`stock_ret - csi800_ret`。
2. `median_label_1m`：横截面中位数相对 label，`alpha_1m - 当月median(alpha_1m)`。
3. `trade_alpha_open_1m`：调仓日 open 到下月调仓日 open 的个股收益，减同期 CSI800 open-to-open 收益。

`alpha_win_5_95`、`alpha_z_1m`、`alpha_rank_centered` 已经回测验证较弱，本 notebook 不再训练，避免继续消耗时间和干扰归因。

导出的 pkl 均兼容现有 V46/V410 executor。


In [ ]:
import os
import gc
import pickle
import warnings
import datetime

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception:
    # Local syntax checks do not have JoinQuant APIs. The rebuild path is only used in JoinQuant research.
    pass

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"

# Data rebuild switch. Keep False for normal label/model experiments.
# Set True inside JoinQuant research when you want to regenerate the V4/V46 monthly dataset from APIs.
REBUILD_DATA = False
USE_REBUILT_DATA_FOR_TRAINING = True
REBUILD_DATA_START = "2019-01-01"
REBUILD_DATA_END_FOR_LABEL = "2026-05-31"
REBUILD_DATA_TAG = "{}_{}".format(REBUILD_DATA_START.replace("-", ""), REBUILD_DATA_END_FOR_LABEL.replace("-", ""))
REBUILD_DATA_OUTPUT_PATH = "train_csi800_factor_v40_data_enhancement_{}.csv".format(REBUILD_DATA_TAG)
REBUILD_MANIFEST_PATH = "train_csi800_factor_v40_data_enhancement_{}_manifest.csv".format(REBUILD_DATA_TAG)
REBUILD_FORCE_OVERWRITE = False
OUT_DIR = "v46_label_compare_direct_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
LABEL_END = "2025-03-31"
REQUIRE_LABEL_END_WITHIN_TRAIN = False  # 保持 V46 legacy_unsealed 口径

BENCHMARK = "000906.XSHG"
TOP_N_CANDIDATES = 30
STOCK_NUM = 10
INDUSTRY_CAP_RATIO = 0.20
CORR_THRESHOLD = 0.70
INNER_VALID_FRAC = 0.20
INNER_VALID_MIN_MONTHS = 6
SEED = 42
FIXED_NUM_BOOST_ROUND = 120

# 该 label 需要聚宽研究环境 get_price。若只想离线复跑原始/median，可改为 False。
BUILD_TRADE_OPEN_LABEL = True
OPEN_PRICE_CHUNK_SIZE = 120

TARGET_SPECS = [
    {
        "target_col": "alpha_1m",
        "tag": "orig_alpha",
        "research_version": "candidate_v46_orig_alpha_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed",
        "model_file": "model_candidate_v46_orig_alpha_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl",
        "target_note": "original V46 label: alpha_1m = stock_return - csi800_return",
    },
    {
        "target_col": "median_label_1m",
        "tag": "median_label",
        "research_version": "candidate_v46_median_label_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed",
        "model_file": "model_candidate_v46_median_label_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl",
        "target_note": "median_label_1m = alpha_1m - monthly_median(alpha_1m), equivalent to stock_return - monthly_median(stock_return)",
    },
    {
        "target_col": "trade_alpha_open_1m",
        "tag": "trade_alpha_open",
        "research_version": "candidate_v46_trade_alpha_open_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed",
        "model_file": "model_candidate_v46_trade_alpha_open_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl",
        "target_note": "stock open-to-open return from rebalance_date to next_date minus CSI800 open-to-open return",
    },
]

print("DATA_PATH =", DATA_PATH)
print("REBUILD_DATA =", REBUILD_DATA, "REBUILD_DATA_OUTPUT_PATH =", REBUILD_DATA_OUTPUT_PATH)
print("OUT_DIR =", OUT_DIR)
print("targets =", [x["target_col"] for x in TARGET_SPECS])


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

CANDIDATE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS


In [ ]:
# =========================
# Optional data rebuild block
# =========================
# Default path: REBUILD_DATA=False, use the existing DATA_PATH CSV.
# Rebuild path: set REBUILD_DATA=True inside JoinQuant research; this will regenerate the monthly CSI800 V4/V46 dataset.

UNIVERSE_NAME = "CSI800"
UNIVERSE_INDEX = "000906.XSHG"
MIN_LISTING_DAYS = 180
V4_DATA_START = REBUILD_DATA_START
V4_DATA_END_FOR_LABEL = REBUILD_DATA_END_FOR_LABEL
V4_DATA_FILE = REBUILD_DATA_OUTPUT_PATH

V4_PRICE_PATH_COLS = [
    "px_ret_5", "px_ret_20", "px_ret_60", "px_ret_120",
    "px_close_to_ma20", "px_close_to_ma60", "px_ma20_to_ma60",
    "px_volatility_20", "px_volatility_60", "px_drawdown_20", "px_drawdown_60", "px_drawdown_120",
    "px_up_day_ratio_20", "px_new_high_distance_60", "px_new_low_distance_60",
    "px_skew_20", "px_kurt_20",
]

TRADE_LIQUIDITY_COLS = [
    "liq_money_mean_20", "liq_money_mean_60", "liq_money_ratio_20_60",
    "liq_volume_mean_20", "liq_volume_ratio_20_60",
    "liq_amplitude_mean_20", "liq_amplitude_mean_60",
    "liq_paused_count_20", "liq_paused_count_60",
    "liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20", "liq_one_price_limit_count_20",
]

CONTEXT_COLS = [
    "ctx_industry_ret_20", "ctx_industry_ret_60",
    "ctx_stock_minus_industry_ret_20", "ctx_stock_minus_industry_ret_60",
    "ctx_stock_rank_industry_ret_20", "ctx_stock_rank_industry_volatility_20",
    "ctx_market_ret_20", "ctx_market_ret_60", "ctx_market_volatility_20",
]

CORE_TEMPORAL_FACTORS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio",
    "Rank1M", "sharpe_ratio_60", "VOSC", "MFI14",
]

def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def require_joinquant_api():
    # Directly probe the JoinQuant API instead of inspecting notebook namespaces.
    try:
        get_trade_days(end_date="2019-01-02", count=1)
    except NameError:
        raise RuntimeError("data rebuild requires JoinQuant research runtime: get_trade_days is not available")
    except Exception:
        # API exists; date/account related errors should surface in the actual caller.
        pass


def get_period_date(period, start_date, end_date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    if len(trade_days) == 0:
        return []
    if period != "M":
        raise ValueError("V4 data pipeline only supports monthly period M")

    dates = []
    last_key = None
    for d in trade_days:
        key = d.strftime("%Y-%m")
        if key != last_key:
            dates.append(d.strftime("%Y-%m-%d"))
            last_key = key
    return dates


def get_previous_trade_date(date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(end_date=date, count=2))
    if len(trade_days) < 2:
        return None
    return trade_days[-2].strftime("%Y-%m-%d")


def delect_stop(stocks, begin_date, n=180):
    stock_list = []
    begin_dt = pd.Timestamp(begin_date).to_pydatetime()
    for stock in stocks:
        info = get_security_info(stock)
        if info is None:
            continue
        if info.start_date <= (begin_dt - datetime.timedelta(days=n)).date():
            stock_list.append(stock)
    return stock_list


def filter_paused_stock_by_date(stock_list, date):
    if len(stock_list) == 0:
        return []
    try:
        paused_df = get_price(
            stock_list,
            end_date=date,
            frequency="daily",
            fields=["paused"],
            count=1,
            skip_paused=False,
            panel=False,
            fill_paused=True,
        )
    except Exception:
        return stock_list

    if paused_df is None or paused_df.empty or "paused" not in paused_df.columns:
        return stock_list

    paused_map = paused_df.groupby("code")["paused"].last()
    return [
        stock for stock in stock_list
        if (stock not in paused_map.index) or (not bool(paused_map.loc[stock]))
    ]


def get_stock(stock_pool, feature_date):
    require_joinquant_api()
    if stock_pool == "CSI800":
        stock_list = get_index_stocks(UNIVERSE_INDEX, feature_date)
    elif stock_pool == "HS300":
        stock_list = get_index_stocks("000300.XSHG", feature_date)
    elif stock_pool == "ZZ1000":
        stock_list = get_index_stocks("000852.XSHG", feature_date)
    elif stock_pool == "A":
        stock_list = get_index_stocks("000985.XSHG", feature_date)
    else:
        raise ValueError("unsupported stock_pool: {}".format(stock_pool))

    if len(stock_list) == 0:
        return []

    st_data = get_extras("is_st", stock_list, count=1, end_date=feature_date)
    if st_data is not None and len(st_data) > 0:
        st_row = st_data.iloc[0]
        stock_list = [
            stock for stock in stock_list
            if (stock not in st_row.index) or pd.isnull(st_row[stock]) or (not bool(st_row[stock]))
        ]

    stock_list = filter_paused_stock_by_date(stock_list, feature_date)
    stock_list = delect_stop(stock_list, feature_date, n=MIN_LISTING_DAYS)
    return stock_list


def get_industry_bucket_map_for_data(stock_list, date):
    if len(stock_list) == 0:
        return {}
    try:
        industry_info = get_industry(stock_list, date=date)
    except Exception:
        return {stock: "UNKNOWN" for stock in stock_list}

    out = {}
    for stock in stock_list:
        info = industry_info.get(stock, {})
        bucket = None
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = info.get(key, None)
            if isinstance(sub, dict):
                bucket = sub.get("industry_code") or sub.get("industry_name")
                if bucket:
                    break
        out[stock] = bucket if bucket else "UNKNOWN"
    return out


def get_factor_data(stock_list, date):
    if len(stock_list) == 0:
        return pd.DataFrame()

    df_factor = pd.DataFrame(index=stock_list)
    for fac_chunk in chunks(BASE_FACTOR_COLS, 20):
        try:
            factor_data = get_factor_values(
                securities=stock_list,
                factors=fac_chunk,
                count=1,
                end_date=date,
            )
        except Exception:
            factor_data = None

        for fac in fac_chunk:
            try:
                if factor_data is not None and fac in factor_data:
                    df_factor[fac] = factor_data[fac].iloc[0, :]
                else:
                    df_factor[fac] = np.nan
            except Exception:
                df_factor[fac] = np.nan
    return df_factor


def calc_ret(close_mat, days):
    if close_mat is None or close_mat.empty or len(close_mat) <= days:
        return pd.Series(index=close_mat.columns if close_mat is not None else [], dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[-days - 1] - 1


def calc_up_day_ratio(ret_mat, days):
    if ret_mat is None or ret_mat.empty:
        return pd.Series(dtype=float)
    return (ret_mat.tail(days) > 0).mean()


def calc_new_low_distance(close_mat, days):
    if close_mat is None or close_mat.empty:
        return pd.Series(dtype=float)
    last_close = close_mat.iloc[-1]
    min_close = close_mat.tail(days).min()
    return last_close / min_close - 1


def get_price_path_and_liquidity_data(stock_list, date, lookback=121, chunk_size=160):
    cols = V4_PRICE_PATH_COLS + TRADE_LIQUIDITY_COLS[:9]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "volume", "money", "paused"],
                count=lookback,
                skip_paused=False,
                fq="pre",
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None

        if price_df is None or price_df.empty:
            out_all.append(out)
            continue

        for col in ["close", "high", "low", "volume", "money", "paused"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        volume_mat = price_df.pivot_table(index="time", columns="code", values="volume").sort_index()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        paused_mat = price_df.pivot_table(index="time", columns="code", values="paused").sort_index()

        ret_mat = close_mat.pct_change()
        last_close = close_mat.iloc[-1]
        ma20 = close_mat.tail(20).mean()
        ma60 = close_mat.tail(60).mean()
        money20 = money_mat.tail(20).mean()
        money60 = money_mat.tail(60).mean()
        volume20 = volume_mat.tail(20).mean()
        volume60 = volume_mat.tail(60).mean()

        out["px_ret_5"] = calc_ret(close_mat, 5)
        out["px_ret_20"] = calc_ret(close_mat, 20)
        out["px_ret_60"] = calc_ret(close_mat, 60)
        out["px_ret_120"] = calc_ret(close_mat, 120)
        out["px_close_to_ma20"] = last_close / ma20 - 1
        out["px_close_to_ma60"] = last_close / ma60 - 1
        out["px_ma20_to_ma60"] = ma20 / ma60 - 1
        out["px_volatility_20"] = ret_mat.tail(20).std()
        out["px_volatility_60"] = ret_mat.tail(60).std()
        out["px_drawdown_20"] = last_close / close_mat.tail(20).max() - 1
        out["px_drawdown_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_drawdown_120"] = last_close / close_mat.tail(120).max() - 1
        out["px_up_day_ratio_20"] = calc_up_day_ratio(ret_mat, 20)
        out["px_new_high_distance_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_new_low_distance_60"] = calc_new_low_distance(close_mat, 60)
        out["px_skew_20"] = ret_mat.tail(20).skew()
        out["px_kurt_20"] = ret_mat.tail(20).kurt()

        out["liq_money_mean_20"] = money20
        out["liq_money_mean_60"] = money60
        out["liq_money_ratio_20_60"] = money20 / money60 - 1
        out["liq_volume_mean_20"] = volume20
        out["liq_volume_ratio_20_60"] = volume20 / volume60 - 1
        out["liq_amplitude_mean_20"] = (high_mat.tail(20) / low_mat.tail(20) - 1).mean()
        out["liq_amplitude_mean_60"] = (high_mat.tail(60) / low_mat.tail(60) - 1).mean()
        out["liq_paused_count_20"] = paused_mat.tail(20).fillna(0).sum()
        out["liq_paused_count_60"] = paused_mat.tail(60).fillna(0).sum()

        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, close_mat, high_mat, low_mat, volume_mat, money_mat, paused_mat, ret_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_limit_state_data(stock_list, date, lookback=20, chunk_size=160):
    cols = [
        "liq_low_money_days_20",
        "liq_limit_up_count_20",
        "liq_limit_down_count_20",
        "liq_one_price_limit_count_20",
    ]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk,
                end_date=date,
                frequency="daily",
                fields=["close", "high", "low", "money", "paused", "high_limit", "low_limit"],
                count=lookback,
                skip_paused=False,
                fq=None,
                panel=False,
                fill_paused=True,
            )
        except Exception:
            price_df = None

        if price_df is None or price_df.empty:
            out_all.append(out)
            continue

        for col in ["close", "high", "low", "money", "paused", "high_limit", "low_limit"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        high_limit_mat = price_df.pivot_table(index="time", columns="code", values="high_limit").sort_index()
        low_limit_mat = price_df.pivot_table(index="time", columns="code", values="low_limit").sort_index()

        money_q20 = money_mat.stack().quantile(0.20) if len(money_mat.stack().dropna()) else np.nan
        out["liq_low_money_days_20"] = (money_mat.tail(20) < money_q20).sum() if not pd.isnull(money_q20) else np.nan

        limit_up = close_mat >= (high_limit_mat * 0.999)
        limit_down = close_mat <= (low_limit_mat * 1.001)
        one_price = (high_mat <= low_mat * 1.0001) & (limit_up | limit_down)
        out["liq_limit_up_count_20"] = limit_up.tail(20).sum()
        out["liq_limit_down_count_20"] = limit_down.tail(20).sum()
        out["liq_one_price_limit_count_20"] = one_price.tail(20).sum()

        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, money_mat, close_mat, high_mat, low_mat, high_limit_mat, low_limit_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list)


def get_market_context(date, benchmark=BENCHMARK, lookback=61):
    out = {
        "ctx_market_ret_20": np.nan,
        "ctx_market_ret_60": np.nan,
        "ctx_market_volatility_20": np.nan,
    }
    try:
        bench_df = get_price(
            benchmark,
            end_date=date,
            frequency="daily",
            fields=["close"],
            count=lookback,
            skip_paused=True,
            fq="pre",
        )
    except Exception:
        bench_df = None

    if bench_df is None or bench_df.empty or "close" not in bench_df.columns:
        return out
    close = bench_df["close"].dropna()
    if len(close) > 20:
        out["ctx_market_ret_20"] = close.iloc[-1] / close.iloc[-21] - 1
        out["ctx_market_volatility_20"] = close.pct_change().tail(20).std()
    if len(close) > 60:
        out["ctx_market_ret_60"] = close.iloc[-1] / close.iloc[-61] - 1
    return out


def attach_industry_context(factor_data, market_context):
    out = factor_data.copy()
    for col, value in market_context.items():
        out[col] = value

    for ret_col, ctx_col in [
        ("px_ret_20", "ctx_industry_ret_20"),
        ("px_ret_60", "ctx_industry_ret_60"),
    ]:
        out[ctx_col] = out.groupby("industry_bucket")[ret_col].transform("mean")

    out["ctx_stock_minus_industry_ret_20"] = out["px_ret_20"] - out["ctx_industry_ret_20"]
    out["ctx_stock_minus_industry_ret_60"] = out["px_ret_60"] - out["ctx_industry_ret_60"]
    out["ctx_stock_rank_industry_ret_20"] = out.groupby("industry_bucket")["px_ret_20"].rank(pct=True)
    out["ctx_stock_rank_industry_volatility_20"] = out.groupby("industry_bucket")["px_volatility_20"].rank(pct=True)
    return out


def get_forward_alpha(stock_list, date, next_date, benchmark):
    if len(stock_list) == 0:
        return pd.Series(dtype=float)

    price_df = get_price(
        stock_list,
        start_date=date,
        end_date=next_date,
        frequency="daily",
        fields=["close"],
        skip_paused=True,
        fq="pre",
        panel=False,
    )
    if price_df is None or price_df.empty:
        return pd.Series(dtype=float)

    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
    if len(close_mat) < 2:
        return pd.Series(dtype=float)
    stock_ret = close_mat.iloc[-1] / close_mat.iloc[1] - 1

    bench_df = get_price(
        benchmark,
        start_date=date,
        end_date=next_date,
        frequency="daily",
        fields=["close"],
        skip_paused=True,
        fq="pre",
    )
    if bench_df is None or bench_df.empty or len(bench_df) < 2:
        return pd.Series(dtype=float)

    bench_ret = bench_df["close"].iloc[-1] / bench_df["close"].iloc[1] - 1
    return stock_ret - bench_ret


def add_core_factor_temporal_features(df):
    out = df.copy()
    out = out.sort_values(["rebalance_date", "stock"]).reset_index(drop=True)
    for factor in CORE_TEMPORAL_FACTORS:
        if factor not in out.columns:
            continue
        rank_col = "tmp_{}_rank".format(factor)
        out[rank_col] = out.groupby("rebalance_date")[factor].rank(pct=True)
        g_stock = out.groupby("stock")[rank_col]
        for lag in [1, 3]:
            col = "ts_{}_rank_chg_{}m".format(factor, lag)
            out[col] = out[rank_col] - g_stock.shift(lag)
        mean_col = "ts_{}_rank_mean_3m".format(factor)
        std_col = "ts_{}_rank_std_3m".format(factor)
        z_col = "ts_{}_rank_z_6m".format(factor)
        out[mean_col] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
        out[std_col] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
        rolling_mean_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
        rolling_std_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).std())
        out[z_col] = (out[rank_col] - rolling_mean_6) / rolling_std_6
        out = out.drop(columns=[rank_col])
    return out


def build_v46_rebuild_dataset():
    require_joinquant_api()
    date_list = get_period_date("M", V4_DATA_START, V4_DATA_END_FOR_LABEL)
    print("V46 rebuild rebalance dates =", len(date_list), "|", V4_DATA_START, "->", V4_DATA_END_FOR_LABEL)

    all_rows = []
    for i, rebalance_date in enumerate(date_list[:-1]):
        next_date = date_list[i + 1]
        feature_date = get_previous_trade_date(rebalance_date)
        if feature_date is None:
            continue

        stock_list = get_stock(UNIVERSE_NAME, feature_date)
        if len(stock_list) == 0:
            continue

        jq_factor_data = get_factor_data(stock_list, feature_date)
        if jq_factor_data is None or jq_factor_data.empty:
            continue

        industry_map = get_industry_bucket_map_for_data(stock_list, feature_date)
        price_liq_data = get_price_path_and_liquidity_data(stock_list, feature_date)
        limit_data = get_limit_state_data(stock_list, feature_date)
        alpha = get_forward_alpha(stock_list, rebalance_date, next_date, BENCHMARK)
        if alpha.empty:
            continue

        factor_data = jq_factor_data.join(price_liq_data, how="left").join(limit_data, how="left")
        factor_data["stock"] = factor_data.index
        factor_data["industry_bucket"] = factor_data["stock"].map(industry_map).fillna("UNKNOWN")
        factor_data = attach_industry_context(factor_data, get_market_context(feature_date, BENCHMARK))
        factor_data["alpha_1m"] = alpha
        factor_data["rebalance_date"] = rebalance_date
        factor_data["feature_date"] = feature_date
        factor_data["next_date"] = next_date
        factor_data = factor_data.dropna(subset=["alpha_1m"]).copy()
        if len(factor_data) < 30:
            continue

        factor_data["alpha_rank_pct"] = factor_data["alpha_1m"].rank(pct=True, method="first")
        all_rows.append(factor_data.reset_index(drop=True))
        print(
            "  rebuilt {}/{} rebalance={} feature={} rows={}".format(
                i + 1, max(1, len(date_list) - 1), rebalance_date, feature_date, len(factor_data)
            )
        )

        del jq_factor_data, price_liq_data, limit_data, alpha, factor_data
        gc.collect()

    if len(all_rows) == 0:
        raise ValueError("V46 data rebuild produced no rows")
    df = pd.concat(all_rows, ignore_index=True)
    df = add_core_factor_temporal_features(df)
    df.to_csv(V4_DATA_FILE, index=False)
    print("V46 data rebuilt rows =", len(df), "saved ->", V4_DATA_FILE)
    return df





def maybe_rebuild_dataset():
    global DATA_PATH
    if not REBUILD_DATA:
        print("skip data rebuild; using DATA_PATH =", DATA_PATH)
        return None

    if os.path.exists(REBUILD_DATA_OUTPUT_PATH) and not REBUILD_FORCE_OVERWRITE:
        print("rebuilt data already exists; skip rebuild:", REBUILD_DATA_OUTPUT_PATH)
        if USE_REBUILT_DATA_FOR_TRAINING:
            DATA_PATH = REBUILD_DATA_OUTPUT_PATH
            print("DATA_PATH switched to existing rebuilt file:", DATA_PATH)
        return pd.read_csv(REBUILD_DATA_OUTPUT_PATH, nrows=5)

    print("rebuilding CSI800 monthly dataset")
    print("  start =", REBUILD_DATA_START, "end_for_label =", REBUILD_DATA_END_FOR_LABEL)
    rebuilt_df = build_v46_rebuild_dataset()

    for _col in ["rebalance_date", "feature_date", "next_date"]:
        if _col in rebuilt_df.columns:
            rebuilt_df[_col] = pd.to_datetime(rebuilt_df[_col])

    meta_cols = ["stock", "rebalance_date", "feature_date", "next_date", "alpha_1m", "alpha_rank_pct", "industry_bucket"]
    feature_cols_rebuilt = [c for c in rebuilt_df.columns if c not in meta_cols]
    date_min = rebuilt_df["rebalance_date"].min() if len(rebuilt_df) else pd.NaT
    date_max = rebuilt_df["rebalance_date"].max() if len(rebuilt_df) else pd.NaT

    rebuild_manifest = pd.DataFrame([{
        "data_file": REBUILD_DATA_OUTPUT_PATH,
        "rows": int(len(rebuilt_df)),
        "months": int(rebuilt_df["rebalance_date"].nunique()) if "rebalance_date" in rebuilt_df.columns else 0,
        "stock_count": int(rebuilt_df["stock"].nunique()) if "stock" in rebuilt_df.columns else 0,
        "feature_count": int(len(feature_cols_rebuilt)),
        "rebalance_date_min": str(date_min.date()) if not pd.isnull(date_min) else "",
        "rebalance_date_max": str(date_max.date()) if not pd.isnull(date_max) else "",
        "universe": UNIVERSE_NAME,
        "universe_index": UNIVERSE_INDEX,
        "benchmark": BENCHMARK,
        "min_listing_days": MIN_LISTING_DAYS,
        "rebuilt_by": "中证800_V46_median_label_direct实验.ipynb",
    }])
    rebuild_manifest.to_csv(REBUILD_MANIFEST_PATH, index=False)

    print("rebuilt rows =", len(rebuilt_df))
    print("rebalance date =", date_min, "->", date_max)
    print("feature count =", len(feature_cols_rebuilt))
    print("saved data ->", REBUILD_DATA_OUTPUT_PATH)
    print("saved manifest ->", REBUILD_MANIFEST_PATH)
    display(rebuild_manifest)

    if USE_REBUILT_DATA_FOR_TRAINING:
        DATA_PATH = REBUILD_DATA_OUTPUT_PATH
        print("DATA_PATH switched to rebuilt file:", DATA_PATH)

    return rebuilt_df


_rebuilt_preview_df = maybe_rebuild_dataset()
if _rebuilt_preview_df is not None:
    print("rebuild preview shape:", _rebuilt_preview_df.shape)


In [ ]:
def unique_keep_order(cols):
    seen, out = set(), []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": a, "b": b}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited, comps = set(), []
    def dfs(x, comp):
        visited.add(x); comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep, remove = [], []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0]); remove.extend(comp[1:])
    return keep, remove


def split_inner_train_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    n_valid = max(INNER_VALID_MIN_MONTHS, int(round(len(months) * INNER_VALID_FRAC)))
    valid_months = set(months[-min(n_valid, max(1, len(months) - 1)):])
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty:
        fit, valid = train_df.copy(), train_df.copy()
    return fit, valid


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = d[target_col].astype(float)
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values


In [ ]:
def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def normalize_price_date(s):
    return pd.to_datetime(s).dt.normalize()


def fetch_stock_open_prices(pairs, chunk_size=OPEN_PRICE_CHUNK_SIZE):
    pairs = pairs[["stock", "date"]].dropna().drop_duplicates().copy()
    pairs["date"] = normalize_price_date(pairs["date"])
    out_parts = []

    for dt, day_pairs in pairs.groupby("date"):
        stock_list = list(day_pairs["stock"].drop_duplicates())
        dt_str = pd.Timestamp(dt).strftime("%Y-%m-%d")
        for stock_chunk in chunks(stock_list, chunk_size):
            try:
                px = get_price(
                    stock_chunk,
                    end_date=dt_str,
                    frequency="daily",
                    fields=["open"],
                    count=1,
                    panel=False,
                    fq="pre",
                    skip_paused=False,
                    fill_paused=True,
                )
            except NameError as err:
                raise RuntimeError("BUILD_TRADE_OPEN_LABEL=True requires JoinQuant get_price; run this cell in JoinQuant research or set BUILD_TRADE_OPEN_LABEL=False")
            except Exception as err:
                print("open price fetch failed:", dt_str, len(stock_chunk), err)
                continue
            if px is None or len(px) == 0:
                continue
            tmp = px.copy()
            if "code" not in tmp.columns:
                tmp = tmp.reset_index()
                if "code" not in tmp.columns and "index" in tmp.columns:
                    tmp = tmp.rename(columns={"index": "code"})
            if "open" not in tmp.columns:
                continue
            tmp = tmp[["code", "open"]].rename(columns={"code": "stock", "open": "open_price"})
            tmp["date"] = pd.Timestamp(dt)
            out_parts.append(tmp)

    if not out_parts:
        raise RuntimeError("no stock open prices fetched; check JoinQuant get_price availability and date range")
    out = pd.concat(out_parts, ignore_index=True)
    out["date"] = normalize_price_date(out["date"])
    out["open_price"] = pd.to_numeric(out["open_price"], errors="coerce")
    return out.drop_duplicates(["stock", "date"], keep="last")


def fetch_benchmark_open_prices(dates, benchmark=BENCHMARK):
    out_parts = []
    for dt in sorted(pd.to_datetime(pd.Series(dates).dropna().unique())):
        dt_str = pd.Timestamp(dt).strftime("%Y-%m-%d")
        try:
            px = get_price(
                benchmark,
                end_date=dt_str,
                frequency="daily",
                fields=["open"],
                count=1,
                panel=False,
                fq="pre",
                skip_paused=False,
                fill_paused=True,
            )
        except NameError:
            raise RuntimeError("BUILD_TRADE_OPEN_LABEL=True requires JoinQuant get_price; run this cell in JoinQuant research or set BUILD_TRADE_OPEN_LABEL=False")
        except Exception as err:
            print("benchmark open fetch failed:", dt_str, err)
            continue
        if px is None or len(px) == 0 or "open" not in px.columns:
            continue
        out_parts.append({"date": pd.Timestamp(dt), "benchmark_open": float(px.iloc[-1]["open"])})
    if not out_parts:
        raise RuntimeError("no benchmark open prices fetched")
    out = pd.DataFrame(out_parts)
    out["date"] = normalize_price_date(out["date"])
    return out.drop_duplicates(["date"], keep="last")


def add_trade_open_label(df):
    out = df.copy()
    start_pairs = out[["stock", "rebalance_date"]].rename(columns={"rebalance_date": "date"})
    end_pairs = out[["stock", "next_date"]].rename(columns={"next_date": "date"})
    stock_px = fetch_stock_open_prices(pd.concat([start_pairs, end_pairs], ignore_index=True))

    entry = stock_px.rename(columns={"date": "rebalance_date", "open_price": "entry_open"})
    exit_ = stock_px.rename(columns={"date": "next_date", "open_price": "exit_open"})
    out = out.merge(entry[["stock", "rebalance_date", "entry_open"]], on=["stock", "rebalance_date"], how="left")
    out = out.merge(exit_[["stock", "next_date", "exit_open"]], on=["stock", "next_date"], how="left")

    bench_dates = pd.concat([out["rebalance_date"], out["next_date"]], ignore_index=True)
    bench_px = fetch_benchmark_open_prices(bench_dates)
    bench_entry = bench_px.rename(columns={"date": "rebalance_date", "benchmark_open": "benchmark_entry_open"})
    bench_exit = bench_px.rename(columns={"date": "next_date", "benchmark_open": "benchmark_exit_open"})
    out = out.merge(bench_entry, on="rebalance_date", how="left")
    out = out.merge(bench_exit, on="next_date", how="left")

    out["trade_return_open_1m"] = out["exit_open"] / out["entry_open"] - 1.0
    out["benchmark_trade_return_open_1m"] = out["benchmark_exit_open"] / out["benchmark_entry_open"] - 1.0
    out["trade_alpha_open_1m"] = out["trade_return_open_1m"] - out["benchmark_trade_return_open_1m"]
    return out.replace([np.inf, -np.inf], np.nan)


def add_label_variants(df):
    out = df.copy()

    # stock_ret - median(stock_ret) == alpha_1m - median(alpha_1m), because benchmark return is constant within month.
    g = out.groupby("rebalance_date")["alpha_1m"]
    out["median_label_1m"] = out["alpha_1m"] - g.transform("median")

    if BUILD_TRADE_OPEN_LABEL:
        out = add_trade_open_label(out)
    else:
        out["trade_alpha_open_1m"] = np.nan
    return out.replace([np.inf, -np.inf], np.nan)


def load_train_df(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        df[col] = pd.to_datetime(df[col]).dt.normalize()
    df["alpha_1m"] = pd.to_numeric(df["alpha_1m"], errors="coerce")
    df = df.dropna(subset=["stock", "rebalance_date", "next_date", "alpha_1m"]).copy()
    df = add_label_variants(df)

    train = df[(df["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (df["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
    if REQUIRE_LABEL_END_WITHIN_TRAIN:
        train = train[train["next_date"] <= pd.Timestamp(LABEL_END)].copy()
    return df, train


df_all, train_df = load_train_df(DATA_PATH)
label_cols = [x["target_col"] for x in TARGET_SPECS]
print("all:", df_all.shape, df_all["rebalance_date"].min(), df_all["rebalance_date"].max())
print("train:", train_df.shape, train_df["rebalance_date"].min(), train_df["rebalance_date"].max(), "months", train_df["rebalance_date"].nunique())
print("label coverage:")
print(train_df[label_cols].notnull().mean())
print(train_df[label_cols].describe())


In [ ]:
fit_df, inner_valid_df = split_inner_train_valid(train_df)
feature_cols, removed_cols = select_features_train_only(fit_df, CANDIDATE_COLS)
print("features:", len(feature_cols), "removed:", len(removed_cols))
print(feature_cols)
print("fixed_num_boost_round:", FIXED_NUM_BOOST_ROUND)


def train_one_target(target_col):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED

    X_train, y_train, fill_values = prepare_xy(train_df, feature_cols, target_col)
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(FIXED_NUM_BOOST_ROUND)),
        valid_sets=[lgb.Dataset(X_train, label=y_train)],
        valid_names=["train"],
        verbose_eval=False,
    )

    X_valid, y_valid, _ = prepare_xy(inner_valid_df, feature_cols, target_col, fill_values)
    valid_pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=FIXED_NUM_BOOST_ROUND)).reshape(-1)
    inner_rank_ic = safe_rank_ic(y_valid, valid_pred)

    return {
        "target_col": target_col,
        "model": model,
        "fill_values": fill_values,
        "fixed_iter": int(FIXED_NUM_BOOST_ROUND),
        "inner_rank_ic": inner_rank_ic,
        "train_rows": int(len(X_train)),
    }


TRAINED_MODELS = {}
for spec in TARGET_SPECS:
    print("training target:", spec["target_col"])
    TRAINED_MODELS[spec["target_col"]] = train_one_target(spec["target_col"])
    print("  fixed_iter:", TRAINED_MODELS[spec["target_col"]]["fixed_iter"], "inner_rank_ic:", TRAINED_MODELS[spec["target_col"]]["inner_rank_ic"])
    gc.collect()


In [ ]:
export_rows = []

for spec in TARGET_SPECS:
    trained = TRAINED_MODELS[spec["target_col"]]
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": spec["research_version"],
        "benchmark": BENCHMARK,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "label_end": LABEL_END,
        "require_label_end_within_train": bool(REQUIRE_LABEL_END_WITHIN_TRAIN),
        "target_col": spec["target_col"],
        "target_note": spec["target_note"],
        "data_file": DATA_PATH,
        "protocol": "v46_direct_label_compare_fixed_iter_full_train",
        "training_policy": "expanding",
        "param_set": "v46_ff10_label_compare",
        "final_role": "v46_label_ablation_same_env",
        "base_params": BASE_PARAMS_FF10,
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(trained["fixed_iter"]),
        "es_best_iter": np.nan,
        "model_iter": int(trained["fixed_iter"]),
        "fixed_iter": int(trained["fixed_iter"]),
        "base_inner_metrics": {"inner_rank_ic": float(trained["inner_rank_ic"]) if not pd.isnull(trained["inner_rank_ic"]) else np.nan},
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": True,
        "uses_time_weight": False,
        "uses_current_valid_for_training": False,
    }

    out_path = os.path.join(OUT_DIR, spec["model_file"])
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=2)

    export_rows.append({
        "tag": spec["tag"],
        "target_col": spec["target_col"],
        "model_file": spec["model_file"],
        "model_path": out_path,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "fixed_iter": trained["fixed_iter"],
        "inner_rank_ic": trained["inner_rank_ic"],
        "train_rows": trained["train_rows"],
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
    })

export_manifest_df = pd.DataFrame(export_rows)
export_manifest_df.to_csv(os.path.join(OUT_DIR, "v46_label_compare_export_manifest.csv"), index=False)
display(export_manifest_df)
print("saved files:")
for row in export_rows:
    print("  " + row["model_path"])


In [ ]:
# Sanity check: exported pkls are compatible with the existing V46/V410 executor.
required = ["objective", "base_model", "base_feature_cols", "base_fill_values", "residual_feature_cols", "residual_fill_values", "overlay_weight", "overlay_mode"]
for spec in TARGET_SPECS:
    path = os.path.join(OUT_DIR, spec["model_file"])
    loaded = pickle.load(open(path, "rb"))
    missing = [k for k in required if k not in loaded]
    print(spec["tag"], "missing keys:", missing)
    print("  objective:", loaded["objective"], "mode:", loaded["overlay_mode"], "features:", len(loaded["base_feature_cols"]), "target:", loaded["target_col"])
